<a href="https://colab.research.google.com/github/Isnob/puzzle_difficulty_anaisys/blob/initial/%D0%9F%D1%80%D0%BE%D0%B5%D0%BA%D1%82.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# =========================
# 1. Загрузка данных
# =========================
df = pd.read_csv("data.csv")  # замени на свой путь
df = df[["answer", "difficulty"]]

# =========================
# 4. НОРМАЛИЗАЦИЯ
# =========================

df["difficulty_norm"] = (
    (df["difficulty"] - df["difficulty"].min()) /
    (df["difficulty"].max() - df["difficulty"].min())
)

# =========================
# 5. ПРИЗНАКИ ОТВЕТА
# =========================

VOWELS = set("аеёиоуыэюя")
RARE = set("фщъёцэ")

def extract_features(text):
    if pd.isna(text):
        return pd.Series([0]*12)

    t = str(text).lower()
    words = t.split()

    len_chars = len(t.replace(" ", ""))
    len_words = len(words)

    vowels = sum(c in VOWELS for c in t)
    consonants = sum(c.isalpha() and c not in VOWELS for c in t)

    rare = sum(c in RARE for c in t)

    unique_chars = len(set(t.replace(" ", "")))

    avg_word_len = len_chars / len_words if len_words else 0

    has_dash = int("-" in t)
    has_digits = int(any(c.isdigit() for c in t))

    return pd.Series([
        len_chars,
        len_words,
        avg_word_len,
        vowels,
        consonants,
        rare,
        unique_chars,
        vowels / len_chars if len_chars else 0,
        rare / len_chars if len_chars else 0,
        unique_chars / len_chars if len_chars else 0,
        has_dash,
        has_digits
    ])

# применяем к колонке с ответом (замени название если нужно)
features = df["answer"].apply(extract_features)

features.columns = [
    "len_chars",
    "len_words",
    "avg_word_len",
    "vowel_count",
    "consonant_count",
    "rare_count",
    "unique_chars",
    "vowel_ratio",
    "rare_ratio",
    "unique_ratio",
    "has_dash",
    "has_digits"
]

df = pd.concat([df, features], axis=1)

# =========================
# 6. ГОТОВО
# =========================

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 326 entries, 0 to 325
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   answer           326 non-null    object 
 1   difficulty       326 non-null    float64
 2   difficulty_norm  326 non-null    float64
 3   len_chars        326 non-null    float64
 4   len_words        326 non-null    float64
 5   avg_word_len     326 non-null    float64
 6   vowel_count      326 non-null    float64
 7   consonant_count  326 non-null    float64
 8   rare_count       326 non-null    float64
 9   unique_chars     326 non-null    float64
 10  vowel_ratio      326 non-null    float64
 11  rare_ratio       326 non-null    float64
 12  unique_ratio     326 non-null    float64
 13  has_dash         326 non-null    float64
 14  has_digits       326 non-null    float64
dtypes: float64(14), object(1)
memory usage: 38.3+ KB
None


In [ ]:
X = df.drop(columns=[
    "answer",           # текст не используем (пока)
    "difficulty",       # исходная метрика
    "difficulty_norm"   # это и есть target
])

y = df["difficulty_norm"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

model = Ridge()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

R2: 0.0028716638488516244
MAE: 0.19839419062451047


In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

R2: -0.06203674772577861
MAE: 0.19967913837496254


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Загружаем данные
df = pd.read_csv("data_with_image_features_fixed.csv")

# 2. Нормализуем сложность
df["difficulty_norm"] = (
    (df["difficulty"] - df["difficulty"].min()) /
    (df["difficulty"].max() - df["difficulty"].min())
)

# 3. Выбираем image features
img_cols = [c for c in df.columns if c.startswith("img_feat_")]

# если есть одна битая строка с ненайденной картинкой, можно оставить как есть
# или удалить:
# df = df[df["resolved_img_path"].notna()].copy()

X = df[img_cols].fillna(0)
y = df["difficulty_norm"]

# 4. Делим на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. PCA + Ridge
model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10, random_state=42)),
    ("ridge", Ridge(alpha=1.0))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

R2: 0.08610167864663687
MAE: 0.1674489657904131


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

# =========================
# 1. Загрузка данных
# =========================
df = pd.read_csv("data_with_image_features_fixed.csv")

# =========================
# 2. Нормализация сложности
# =========================
df["difficulty_norm"] = (
    (df["difficulty"] - df["difficulty"].min()) /
    (df["difficulty"].max() - df["difficulty"].min())
)

# =========================
# 3. Удаляем строки без картинки (опционально, но лучше)
# =========================
if "resolved_img_path" in df.columns:
    df = df[df["resolved_img_path"].notna()].copy()

# =========================
# 4. Выбор признаков
# =========================
text_cols = [
    "len_chars",
    "len_words",
    "avg_word_len",
    "vowel_count",
    "consonant_count",
    "rare_count",
    "unique_chars",
    "vowel_ratio",
    "rare_ratio",
    "unique_ratio",
    "has_dash",
    "has_digits"
]

img_cols = [c for c in df.columns if c.startswith("img_feat_")]

X = df[text_cols + img_cols].copy()
y = df["difficulty_norm"].copy()

# на всякий случай
X = X.fillna(0)

# =========================
# 5. Train / Test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 6. Предобработка
# =========================
preprocessor = ColumnTransformer([
    ("img", Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=50, random_state=42))
    ]), img_cols),

    ("text", StandardScaler(), text_cols)
])

# =========================
# 7. Pipeline
# =========================
pipeline = Pipeline([
    ("prep", preprocessor),
    ("ridge", Ridge())
])

# =========================
# 8. Подбор alpha
# =========================
param_grid = {
    "ridge__alpha": [0.01, 0.1, 1, 10, 50, 100]
}

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid.fit(X_train, y_train)

# =========================
# 9. Лучшая модель
# =========================
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Лучший alpha:", grid.best_params_["ridge__alpha"])
print("CV best R2:", grid.best_score_)
print("Test R2:", r2_score(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))

KeyError: "['len_chars', 'len_words', 'avg_word_len', 'vowel_count', 'consonant_count', 'rare_count', 'unique_chars', 'vowel_ratio', 'rare_ratio', 'unique_ratio', 'has_dash', 'has_digits'] not in index"

# С картинками

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# =========================================
# 1. Загрузка датасета
# =========================================
df = pd.read_csv("data_with_image_features_fixed.csv")

# Нормализуем target, если его ещё нет
df["difficulty_norm"] = (
    (df["difficulty"] - df["difficulty"].min()) /
    (df["difficulty"] - df["difficulty"].min()).max()
)

# Убираем строку без найденной картинки, если такая есть
if "resolved_img_path" in df.columns:
    df = df[df["resolved_img_path"].notna()].copy()

# =========================================
# 2. Выделяем image features
# =========================================
img_cols = [c for c in df.columns if c.startswith("img_feat_")]
y = df["difficulty_norm"].copy()

print("Размер датасета:", df.shape)
print("Число image-признаков:", len(img_cols))

# =========================================
# 3. BASELINE: только картинки
# =========================================
X_img = df[img_cols].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X_img, y, test_size=0.2, random_state=42
)

img_model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=50, random_state=42)),
    ("ridge", Ridge(alpha=1.0))
])

img_model.fit(X_train, y_train)
y_pred_img = img_model.predict(X_test)

print("\n=== Только image features ===")
print("R2:", r2_score(y_test, y_pred_img))
print("MAE:", mean_absolute_error(y_test, y_pred_img))

# =========================================
# 4. Считаем текстовые признаки ответа
# =========================================
VOWELS = set("аеёиоуыэюя")
RARE = set("фщъёцэ")

def extract_text_features(text):
    if pd.isna(text):
        return pd.Series([0] * 12)

    t = str(text).lower().strip()
    words = t.split()

    len_chars = len(t.replace(" ", ""))
    len_words = len(words)
    avg_word_len = len_chars / len_words if len_words else 0

    vowel_count = sum(c in VOWELS for c in t)
    consonant_count = sum(c.isalpha() and c not in VOWELS for c in t)
    rare_count = sum(c in RARE for c in t)
    unique_chars = len(set(t.replace(" ", ""))) if len_chars > 0 else 0

    vowel_ratio = vowel_count / len_chars if len_chars else 0
    rare_ratio = rare_count / len_chars if len_chars else 0
    unique_ratio = unique_chars / len_chars if len_chars else 0

    has_dash = int("-" in t)
    has_digits = int(any(c.isdigit() for c in t))

    return pd.Series([
        len_chars,
        len_words,
        avg_word_len,
        vowel_count,
        consonant_count,
        rare_count,
        unique_chars,
        vowel_ratio,
        rare_ratio,
        unique_ratio,
        has_dash,
        has_digits
    ])

text_features = df["answer"].apply(extract_text_features)
text_features.columns = [
    "len_chars",
    "len_words",
    "avg_word_len",
    "vowel_count",
    "consonant_count",
    "rare_count",
    "unique_chars",
    "vowel_ratio",
    "rare_ratio",
    "unique_ratio",
    "has_dash",
    "has_digits"
]

df = pd.concat([df, text_features], axis=1)

text_cols = list(text_features.columns)

print("\nТекстовые признаки добавлены:")
print(text_cols)

# =========================================
# 5. Картинки + текстовые признаки
# =========================================
X_all = df[img_cols + text_cols].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer([
    ("img", Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=50, random_state=42))
    ]), img_cols),
    ("text", StandardScaler(), text_cols)
])

all_model = Pipeline([
    ("prep", preprocessor),
    ("ridge", Ridge(alpha=1.0))
])

all_model.fit(X_train, y_train)
y_pred_all = all_model.predict(X_test)

print("\n=== Image features + text features ===")
print("R2:", r2_score(y_test, y_pred_all))
print("MAE:", mean_absolute_error(y_test, y_pred_all))

# =========================================
# 6. Gradient Boosting на объединённых признаках
# =========================================
gbr_model = Pipeline([
    ("prep", preprocessor),
    ("gbr", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

gbr_model.fit(X_train, y_train)
y_pred_gbr = gbr_model.predict(X_test)

print("\n=== Gradient Boosting (image + text) ===")
print("R2:", r2_score(y_test, y_pred_gbr))
print("MAE:", mean_absolute_error(y_test, y_pred_gbr))

Размер датасета: (326, 527)
Число image-признаков: 512

=== Только image features ===
R2: 0.0004085324995657391
MAE: 0.20453879700303562

Текстовые признаки добавлены:
['len_chars', 'len_words', 'avg_word_len', 'vowel_count', 'consonant_count', 'rare_count', 'unique_chars', 'vowel_ratio', 'rare_ratio', 'unique_ratio', 'has_dash', 'has_digits']

=== Image features + text features ===
R2: -0.00953886847695995
MAE: 0.19858832870320967

=== Gradient Boosting (image + text) ===
R2: 0.038575419204054606
MAE: 0.19250755763294752


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# =========================================================
# 1. Предполагается, что df уже существует из прошлой ячейки
# =========================================================
print("Размер df:", df.shape)

if "difficulty_norm" not in df.columns:
    df["difficulty_norm"] = (
        (df["difficulty"] - df["difficulty"].min()) /
        (df["difficulty"].max() - df["difficulty"].min())
    )

if "resolved_img_path" in df.columns:
    df = df[df["resolved_img_path"].notna()].copy()

img_cols = [c for c in df.columns if c.startswith("img_feat_")]
y = df["difficulty_norm"].copy()

print("Число image-признаков:", len(img_cols))

# =========================================================
# 2. BASELINE: только имеющиеся image features
#    Подберём несколько значений PCA и alpha
# =========================================================
best_img_result = None

for n_comp in [5, 10, 20, 30]:
    for alpha in [0.01, 0.1, 1, 10, 50, 100]:
        X_img = df[img_cols].fillna(0)

        X_train, X_test, y_train, y_test = train_test_split(
            X_img, y, test_size=0.2, random_state=42
        )

        model = Pipeline([
            ("pca", PCA(n_components=n_comp, random_state=42)),
            ("ridge", Ridge(alpha=alpha))
        ])

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        result = {
            "pca": n_comp,
            "alpha": alpha,
            "r2": r2_score(y_test, y_pred),
            "mae": mean_absolute_error(y_test, y_pred)
        }

        if (best_img_result is None) or (result["r2"] > best_img_result["r2"]):
            best_img_result = result

print("\n=== Только имеющиеся image features ===")
print(best_img_result)

# =========================================================
# 3. Считаем текстовые признаки ответа, если их ещё нет
# =========================================================
VOWELS = set("аеёиоуыэюя")
RARE = set("фщъёцэ")

def extract_text_features(text):
    if pd.isna(text):
        return pd.Series([0] * 12)

    t = str(text).lower().strip()
    words = t.split()

    len_chars = len(t.replace(" ", ""))
    len_words = len(words)
    avg_word_len = len_chars / len_words if len_words else 0

    vowel_count = sum(c in VOWELS for c in t)
    consonant_count = sum(c.isalpha() and c not in VOWELS for c in t)
    rare_count = sum(c in RARE for c in t)
    unique_chars = len(set(t.replace(" ", ""))) if len_chars > 0 else 0

    vowel_ratio = vowel_count / len_chars if len_chars else 0
    rare_ratio = rare_count / len_chars if len_chars else 0
    unique_ratio = unique_chars / len_chars if len_chars else 0

    has_dash = int("-" in t)
    has_digits = int(any(c.isdigit() for c in t))

    return pd.Series([
        len_chars,
        len_words,
        avg_word_len,
        vowel_count,
        consonant_count,
        rare_count,
        unique_chars,
        vowel_ratio,
        rare_ratio,
        unique_ratio,
        has_dash,
        has_digits
    ])

text_cols = [
    "len_chars",
    "len_words",
    "avg_word_len",
    "vowel_count",
    "consonant_count",
    "rare_count",
    "unique_chars",
    "vowel_ratio",
    "rare_ratio",
    "unique_ratio",
    "has_dash",
    "has_digits"
]

if not all(col in df.columns for col in text_cols):
    text_features = df["answer"].apply(extract_text_features)
    text_features.columns = text_cols
    df = pd.concat([df, text_features], axis=1)

# =========================================================
# 4. image features + text features
#    Тоже подберём PCA и alpha
# =========================================================
best_all_result = None

X_all = df[img_cols + text_cols].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=42
)

for n_comp in [5, 10, 20, 30]:
    for alpha in [0.01, 0.1, 1, 10, 50, 100]:
        preprocessor = ColumnTransformer([
            ("img", Pipeline([
                ("pca", PCA(n_components=n_comp, random_state=42))
            ]), img_cols),
            ("text", StandardScaler(), text_cols)
        ])

        model = Pipeline([
            ("prep", preprocessor),
            ("ridge", Ridge(alpha=alpha))
        ])

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        result = {
            "pca": n_comp,
            "alpha": alpha,
            "r2": r2_score(y_test, y_pred),
            "mae": mean_absolute_error(y_test, y_pred)
        }

        if (best_all_result is None) or (result["r2"] > best_all_result["r2"]):
            best_all_result = result

print("\n=== Имеющиеся image features + text features ===")
print(best_all_result)

# =========================================================
# 5. Gradient Boosting на image + text
# =========================================================
best_gbr_result = None

for n_comp in [5, 10, 20, 30]:
    for n_estimators in [50, 100, 200]:
        for learning_rate in [0.03, 0.05, 0.1]:
            for max_depth in [2, 3]:
                preprocessor = ColumnTransformer([
                    ("img", Pipeline([
                        ("pca", PCA(n_components=n_comp, random_state=42))
                    ]), img_cols),
                    ("text", StandardScaler(), text_cols)
                ])

                model = Pipeline([
                    ("prep", preprocessor),
                    ("gbr", GradientBoostingRegressor(
                        n_estimators=n_estimators,
                        learning_rate=learning_rate,
                        max_depth=max_depth,
                        random_state=42
                    ))
                ])

                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)

                result = {
                    "pca": n_comp,
                    "n_estimators": n_estimators,
                    "learning_rate": learning_rate,
                    "max_depth": max_depth,
                    "r2": r2_score(y_test, y_pred),
                    "mae": mean_absolute_error(y_test, y_pred)
                }

                if (best_gbr_result is None) or (result["r2"] > best_gbr_result["r2"]):
                    best_gbr_result = result

print("\n=== Gradient Boosting на имеющихся image + text features ===")
print(best_gbr_result)

Размер df: (326, 546)
Число image-признаков: 512

=== Только имеющиеся image features ===
{'pca': 20, 'alpha': 100, 'r2': 0.057806489695204855, 'mae': 0.1961477628508167}

=== Имеющиеся image features + text features ===
{'pca': 20, 'alpha': 1, 'r2': 0.013199066330057407, 'mae': 0.1935752981096779}

=== Gradient Boosting на имеющихся image + text features ===
{'pca': 10, 'n_estimators': 50, 'learning_rate': 0.1, 'max_depth': 2, 'r2': 0.06637286485242111, 'mae': 0.1912862331188349}


ЛУЧШИЙ РЕЗУЛЬТАТ

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import pandas as pd

# 1. Если difficulty_norm вдруг не посчитан
if "difficulty_norm" not in df.columns:
    df["difficulty_norm"] = (
        (df["difficulty"] - df["difficulty"].min()) /
        (df["difficulty"].max() - df["difficulty"].min())
    )

# 2. Если есть строки без картинки — убираем
if "resolved_img_path" in df.columns:
    df_bin = df[df["resolved_img_path"].notna()].copy()
else:
    df_bin = df.copy()

# 3. Берём только image-признаки
img_cols = [c for c in df_bin.columns if c.startswith("img_feat_")]
X = df_bin[img_cols].fillna(0)

# 4. Бинаризация по медиане
threshold = df_bin["difficulty_norm"].median()
y_reg = df_bin["difficulty_norm"]
y_bin = (y_reg >= threshold).astype(int)

print("Порог (медиана):", threshold)
print("Класс 0 (простые):", (y_bin == 0).sum())
print("Класс 1 (сложные):", (y_bin == 1).sum())

# 5. Train/test split
X_train, X_test, y_train_bin, y_test_bin, y_train_reg, y_test_reg = train_test_split(
    X, y_bin, y_reg, test_size=0.2, random_state=42, stratify=y_bin
)

# 6. Твоя лучшая регрессионная модель: PCA=20 + Ridge(alpha=100)
model = Pipeline([
    ("pca", PCA(n_components=20, random_state=42)),
    ("ridge", Ridge(alpha=100))
])

# обучаем на непрерывной сложности
model.fit(X_train, y_train_reg)

# предсказываем непрерывное значение и переводим в класс
y_pred_reg = model.predict(X_test)
y_pred_bin = (y_pred_reg >= threshold).astype(int)

# 7. Метрики классификации
acc = accuracy_score(y_test_bin, y_pred_bin)
prec = precision_score(y_test_bin, y_pred_bin, zero_division=0)
rec = recall_score(y_test_bin, y_pred_bin, zero_division=0)
f1 = f1_score(y_test_bin, y_pred_bin, zero_division=0)

print("\n=== Бинарная классификация сложности ===")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1-score:", f1)

print("\nConfusion matrix:")
cm = confusion_matrix(y_test_bin, y_pred_bin)
print(cm)

print("\nClassification report:")
print(classification_report(y_test_bin, y_pred_bin, zero_division=0))

# 8. Для удобства — таблица сравнения
results_df = pd.DataFrame({
    "y_true_reg": y_test_reg.values,
    "y_pred_reg": y_pred_reg,
    "y_true_bin": y_test_bin.values,
    "y_pred_bin": y_pred_bin
})

print("\nПримеры предсказаний:")
print(results_df.head(10))

Порог (медиана): 0.44122170951439244
Класс 0 (простые): 163
Класс 1 (сложные): 163

=== Бинарная классификация сложности ===
Accuracy: 0.6212121212121212
Precision: 0.6052631578947368
Recall: 0.696969696969697
F1-score: 0.647887323943662

Confusion matrix:
[[18 15]
 [10 23]]

Classification report:
              precision    recall  f1-score   support

           0       0.64      0.55      0.59        33
           1       0.61      0.70      0.65        33

    accuracy                           0.62        66
   macro avg       0.62      0.62      0.62        66
weighted avg       0.62      0.62      0.62        66


Примеры предсказаний:
   y_true_reg  y_pred_reg  y_true_bin  y_pred_bin
0    0.105911    0.413101           0           0
1    0.124368    0.358224           0           0
2    0.748407    0.510349           1           1
3    0.167436    0.437738           0           0
4    0.460998    0.447689           1           1
5    0.509119    0.480027           1           1


In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np

# 1. Подготовка
df_cls = df.copy()

if "difficulty_norm" not in df_cls.columns:
    df_cls["difficulty_norm"] = (
        (df_cls["difficulty"] - df_cls["difficulty"].min()) /
        (df_cls["difficulty"].max() - df_cls["difficulty"].min())
    )

if "resolved_img_path" in df_cls.columns:
    df_cls = df_cls[df_cls["resolved_img_path"].notna()].copy()

img_cols = [c for c in df_cls.columns if c.startswith("img_feat_")]

# 2. Бинаризация по медиане
threshold = df_cls["difficulty_norm"].median()
df_cls["target_bin"] = (df_cls["difficulty_norm"] >= threshold).astype(int)

X = df_cls[img_cols].fillna(0)
y = df_cls["target_bin"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. Модели и параметры
models = {
    "LogisticRegression": (
        Pipeline([
            ("pca", PCA(random_state=42)),
            ("clf", LogisticRegression(max_iter=5000))
        ]),
        {
            "pca__n_components": [5, 10, 20, 30, 50],
            "clf__C": [0.01, 0.1, 1, 10, 100]
        }
    ),
    "GradientBoostingClassifier": (
        Pipeline([
            ("pca", PCA(random_state=42)),
            ("clf", GradientBoostingClassifier(random_state=42))
        ]),
        {
            "pca__n_components": [5, 10, 20, 30],
            "clf__n_estimators": [50, 100, 200],
            "clf__learning_rate": [0.03, 0.05, 0.1],
            "clf__max_depth": [2, 3]
        }
    ),
    "RandomForestClassifier": (
        Pipeline([
            ("pca", PCA(random_state=42)),
            ("clf", RandomForestClassifier(random_state=42))
        ]),
        {
            "pca__n_components": [5, 10, 20, 30],
            "clf__n_estimators": [100, 200],
            "clf__max_depth": [3, 5, None]
        }
    )
}

results = []

for name, (pipe, params) in models.items():
    grid = GridSearchCV(
        estimator=pipe,
        param_grid=params,
        scoring="f1",
        cv=cv,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    results.append({
        "model": name,
        "best_params": grid.best_params_,
        "cv_best_f1": grid.best_score_,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values("test_f1", ascending=False)
print(results_df)

# 4. Лучшая модель
best_name = results_df.iloc[0]["model"]
print("\nЛучшая модель:", best_name)

best_pipe, best_params = models[best_name]
best_grid = GridSearchCV(
    estimator=best_pipe,
    param_grid=best_params,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)
best_grid.fit(X_train, y_train)

best_model = best_grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

                        model  \
0          LogisticRegression   
1  GradientBoostingClassifier   
2      RandomForestClassifier   

                                         best_params  cv_best_f1  \
0            {'clf__C': 0.1, 'pca__n_components': 5}    0.556176   
1  {'clf__learning_rate': 0.1, 'clf__max_depth': ...    0.599286   
2  {'clf__max_depth': 5, 'clf__n_estimators': 100...    0.571412   

   test_accuracy   test_f1  test_balanced_accuracy  
0       0.636364  0.666667                0.636364  
1       0.530303  0.563380                0.530303  
2       0.484848  0.484848                0.484848  

Лучшая модель: LogisticRegression

Classification report:
              precision    recall  f1-score   support

           0       0.67      0.55      0.60        33
           1       0.62      0.73      0.67        33

    accuracy                           0.64        66
   macro avg       0.64      0.64      0.63        66
weighted avg       0.64      0.64      0.63        

In [ ]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np

# =========================================================
# 1. Подготовка
# =========================================================
df_cls = df.copy()

if "difficulty_norm" not in df_cls.columns:
    df_cls["difficulty_norm"] = (
        (df_cls["difficulty"] - df_cls["difficulty"].min()) /
        (df_cls["difficulty"].max() - df_cls["difficulty"].min())
    )

if "resolved_img_path" in df_cls.columns:
    df_cls = df_cls[df_cls["resolved_img_path"].notna()].copy()

img_cols = [c for c in df_cls.columns if c.startswith("img_feat_")]

# =========================================================
# 2. Оставляем только явные классы
#    нижние 40% = простые
#    верхние 40% = сложные
#    средние 20% удаляем
# =========================================================
q_low = df_cls["difficulty_norm"].quantile(0.4)
q_high = df_cls["difficulty_norm"].quantile(0.6)

df_easy = df_cls[df_cls["difficulty_norm"] <= q_low].copy()
df_hard = df_cls[df_cls["difficulty_norm"] >= q_high].copy()

df_extreme = pd.concat([df_easy, df_hard], axis=0).copy()
df_extreme["target_bin"] = (df_extreme["difficulty_norm"] >= q_high).astype(int)

print("Порог простых (40%):", q_low)
print("Порог сложных (60%):", q_high)
print("Размер новой выборки:", df_extreme.shape)
print("Простые:", (df_extreme["target_bin"] == 0).sum())
print("Сложные:", (df_extreme["target_bin"] == 1).sum())

# =========================================================
# 3. X и y
# =========================================================
X = df_extreme[img_cols].fillna(0)
y = df_extreme["target_bin"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# =========================================================
# 4. Модели
# =========================================================
models = {
    "LogisticRegression": (
        Pipeline([
            ("pca", PCA(random_state=42)),
            ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))
        ]),
        {
            "pca__n_components": [3, 5, 10, 20, 30],
            "clf__C": [0.01, 0.1, 1, 10, 100]
        }
    ),
    "GradientBoostingClassifier": (
        Pipeline([
            ("pca", PCA(random_state=42)),
            ("clf", GradientBoostingClassifier(random_state=42))
        ]),
        {
            "pca__n_components": [3, 5, 10, 20],
            "clf__n_estimators": [50, 100, 200],
            "clf__learning_rate": [0.03, 0.05, 0.1],
            "clf__max_depth": [2, 3]
        }
    ),
    "RandomForestClassifier": (
        Pipeline([
            ("pca", PCA(random_state=42)),
            ("clf", RandomForestClassifier(random_state=42, class_weight="balanced"))
        ]),
        {
            "pca__n_components": [3, 5, 10, 20],
            "clf__n_estimators": [100, 200],
            "clf__max_depth": [3, 5, None]
        }
    )
}

results = []

for name, (pipe, params) in models.items():
    grid = GridSearchCV(
        estimator=pipe,
        param_grid=params,
        scoring="f1",
        cv=cv,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    results.append({
        "model": name,
        "best_params": grid.best_params_,
        "cv_best_f1": grid.best_score_,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values("test_f1", ascending=False)

print("\nСравнение моделей:")
print(results_df)

# =========================================================
# 5. Лучшая модель
# =========================================================
best_name = results_df.iloc[0]["model"]
print("\nЛучшая модель:", best_name)

best_pipe, best_params = models[best_name]
best_grid = GridSearchCV(
    estimator=best_pipe,
    param_grid=best_params,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)
best_grid.fit(X_train, y_train)

best_model = best_grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Порог простых (40%): 0.38233355306526035
Порог сложных (60%): 0.5104372665348276
Размер новой выборки: (262, 547)
Простые: 131
Сложные: 131

Сравнение моделей:
                        model  \
0          LogisticRegression   
2      RandomForestClassifier   
1  GradientBoostingClassifier   

                                         best_params  cv_best_f1  \
0          {'clf__C': 0.01, 'pca__n_components': 20}    0.584494   
2  {'clf__max_depth': 5, 'clf__n_estimators': 200...    0.567482   
1  {'clf__learning_rate': 0.1, 'clf__max_depth': ...    0.578191   

   test_accuracy   test_f1  test_balanced_accuracy  
0       0.584906  0.541667                0.583333  
2       0.566038  0.510638                0.564103  
1       0.452830  0.382979                0.450855  

Лучшая модель: LogisticRegression

Classification report:
              precision    recall  f1-score   support

           0       0.58      0.67      0.62        27
           1       0.59      0.50      0.54        26


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, classification_report, confusion_matrix

# =========================================================
# 1. Подготовка данных
# =========================================================
df_cls = df.copy()

if "difficulty_norm" not in df_cls.columns:
    df_cls["difficulty_norm"] = (
        (df_cls["difficulty"] - df_cls["difficulty"].min()) /
        (df_cls["difficulty"].max() - df_cls["difficulty"].min())
    )

if "resolved_img_path" in df_cls.columns:
    df_cls = df_cls[df_cls["resolved_img_path"].notna()].copy()

img_cols = [c for c in df_cls.columns if c.startswith("img_feat_")]

# бинаризация по медиане
threshold = df_cls["difficulty_norm"].median()
df_cls["target_bin"] = (df_cls["difficulty_norm"] >= threshold).astype(int)

X = df_cls[img_cols].fillna(0)
y = df_cls["target_bin"]

print("Размер выборки:", X.shape)
print("Класс 0:", (y == 0).sum())
print("Класс 1:", (y == 1).sum())
print("Порог:", threshold)

# =========================================================
# 2. Train / test split
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# =========================================================
# 3. Модели для честного сравнения
# =========================================================
models = {
    "LogisticRegression": (
        Pipeline([
            ("pca", PCA(random_state=42)),
            ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))
        ]),
        {
            "pca__n_components": [3, 5, 10, 20, 30, 50],
            "clf__C": [0.01, 0.1, 1, 10, 100]
        }
    ),
    "SVC": (
        Pipeline([
            ("pca", PCA(random_state=42)),
            ("clf", SVC(probability=True, class_weight="balanced"))
        ]),
        {
            "pca__n_components": [3, 5, 10, 20, 30],
            "clf__C": [0.1, 1, 10],
            "clf__kernel": ["rbf", "linear"]
        }
    )
}

results = []

for name, (pipe, params) in models.items():
    grid = GridSearchCV(
        estimator=pipe,
        param_grid=params,
        scoring="f1",
        cv=cv,
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_

    # CV-оценка на train
    cv_f1 = cross_val_score(best_model, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1).mean()
    cv_acc = cross_val_score(best_model, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1).mean()

    # test
    y_pred = best_model.predict(X_test)

    results.append({
        "model": name,
        "best_params": grid.best_params_,
        "cv_f1": cv_f1,
        "cv_accuracy": cv_acc,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, y_pred)
    })

results_df = pd.DataFrame(results).sort_values("test_f1", ascending=False)
print("\nСравнение моделей:")
print(results_df)

# =========================================================
# 4. Лучшая модель
# =========================================================
best_name = results_df.iloc[0]["model"]
print("\nЛучшая модель:", best_name)

best_pipe, best_params = models[best_name]
best_grid = GridSearchCV(
    estimator=best_pipe,
    param_grid=best_params,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)
best_grid.fit(X_train, y_train)

best_model = best_grid.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("\nЛучшие параметры:")
print(best_grid.best_params_)

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

# =========================================================
# 5. Уверенные предсказания
# =========================================================
# Иногда модель сильно лучше работает на уверенных кейсах
confidence = np.abs(y_proba - 0.5)
mask_confident = confidence >= 0.10   # можно попробовать 0.05 / 0.10 / 0.15

if mask_confident.sum() > 0:
    y_test_conf = y_test.iloc[mask_confident]
    y_pred_conf = y_pred[mask_confident]

    print("\n=== Только уверенные предсказания ===")
    print("Покрытие:", mask_confident.mean())
    print("Accuracy:", accuracy_score(y_test_conf, y_pred_conf))
    print("F1:", f1_score(y_test_conf, y_pred_conf))
    print("Balanced accuracy:", balanced_accuracy_score(y_test_conf, y_pred_conf))
else:
    print("\nНет уверенных предсказаний при текущем пороге confidence.")

Размер выборки: (326, 512)
Класс 0: 163
Класс 1: 163
Порог: 0.44122170951439244

Сравнение моделей:
                model                                        best_params  \
0  LogisticRegression            {'clf__C': 0.1, 'pca__n_components': 5}   
1                 SVC  {'clf__C': 0.1, 'clf__kernel': 'rbf', 'pca__n_...   

      cv_f1  cv_accuracy  test_accuracy   test_f1  test_balanced_accuracy  
0  0.556176     0.569231       0.636364  0.666667                0.636364  
1  0.594582     0.557692       0.560606  0.623377                0.560606  

Лучшая модель: LogisticRegression

Лучшие параметры:
{'clf__C': 0.1, 'pca__n_components': 5}

Classification report:
              precision    recall  f1-score   support

           0       0.67      0.55      0.60        33
           1       0.62      0.73      0.67        33

    accuracy                           0.64        66
   macro avg       0.64      0.64      0.63        66
weighted avg       0.64      0.64      0.63        66

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# =========================================
# 1. Подготовка данных
# =========================================
df_best = df.copy()

if "difficulty_norm" not in df_best.columns:
    df_best["difficulty_norm"] = (
        (df_best["difficulty"] - df_best["difficulty"].min()) /
        (df_best["difficulty"].max() - df_best["difficulty"].min())
    )

# если есть столбец с найденными путями к картинкам — убираем пустые
if "resolved_img_path" in df_best.columns:
    df_best = df_best[df_best["resolved_img_path"].notna()].copy()

# берём только image-признаки
img_cols = [c for c in df_best.columns if c.startswith("img_feat_")]

X = df_best[img_cols].fillna(0)

# =========================================
# 2. Бинаризация по медиане
# =========================================
threshold = df_best["difficulty_norm"].median()
y = (df_best["difficulty_norm"] >= threshold).astype(int)

print("Порог:", threshold)
print("Класс 0 (простые):", (y == 0).sum())
print("Класс 1 (сложные):", (y == 1).sum())

# =========================================
# 3. Разделение на train/test
# =========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================================
# 4. Лучшая модель
# =========================================
best_model = Pipeline([
    ("pca", PCA(n_components=5, random_state=42)),
    ("clf", LogisticRegression(
        C=0.1,
        max_iter=5000,
        class_weight="balanced"
    ))
])

best_model.fit(X_train, y_train)

# =========================================
# 5. Предсказание и метрики
# =========================================
y_pred = best_model.predict(X_test)

print("\n=== Лучшая модель ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Порог: 0.44122170951439244
Класс 0 (простые): 163
Класс 1 (сложные): 163

=== Лучшая модель ===
Accuracy: 0.6363636363636364
F1-score: 0.6666666666666666
Balanced accuracy: 0.6363636363636364

Classification report:
              precision    recall  f1-score   support

           0       0.67      0.55      0.60        33
           1       0.62      0.73      0.67        33

    accuracy                           0.64        66
   macro avg       0.64      0.64      0.63        66
weighted avg       0.64      0.64      0.63        66

Confusion matrix:
[[18 15]
 [ 9 24]]


# С эффективными признаками

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)
import pandas as pd

# 1. Загружаем новый датасет
df_eff = pd.read_csv("data_with_efficientnet_features.csv")

# 2. Нормализуем сложность
if "difficulty_norm" not in df_eff.columns:
    df_eff["difficulty_norm"] = (
        (df_eff["difficulty"] - df_eff["difficulty"].min()) /
        (df_eff["difficulty"].max() - df_eff["difficulty"].min())
    )

# 3. Убираем строки без найденной картинки
if "resolved_img_path" in df_eff.columns:
    df_eff = df_eff[df_eff["resolved_img_path"].notna()].copy()

# 4. Берём только EfficientNet-признаки
eff_cols = [c for c in df_eff.columns if c.startswith("effnet_feat_")]
X = df_eff[eff_cols].fillna(0)

# 5. Бинаризация по медиане
threshold = df_eff["difficulty_norm"].median()
y = (df_eff["difficulty_norm"] >= threshold).astype(int)

print("Порог:", threshold)
print("Класс 0:", (y == 0).sum())
print("Класс 1:", (y == 1).sum())
print("Число effnet-признаков:", len(eff_cols))

# 6. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 7. Та же лучшая модель, но на новых признаках
model_eff = Pipeline([
    ("pca", PCA(n_components=5, random_state=42)),
    ("clf", LogisticRegression(
        C=0.1,
        max_iter=5000,
        class_weight="balanced"
    ))
])

model_eff.fit(X_train, y_train)
y_pred = model_eff.predict(X_test)

# 8. Метрики
print("\n=== LogisticRegression + PCA(5) на EfficientNet features ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Порог: 0.44122170951439244
Класс 0: 163
Класс 1: 163
Число effnet-признаков: 1280

=== LogisticRegression + PCA(5) на EfficientNet features ===
Accuracy: 0.5606060606060606
F1-score: 0.5797101449275363
Balanced accuracy: 0.5606060606060606

Classification report:
              precision    recall  f1-score   support

           0       0.57      0.52      0.54        33
           1       0.56      0.61      0.58        33

    accuracy                           0.56        66
   macro avg       0.56      0.56      0.56        66
weighted avg       0.56      0.56      0.56        66

Confusion matrix:
[[17 16]
 [13 20]]


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, classification_report, confusion_matrix
import pandas as pd

df_eff = pd.read_csv("data_with_efficientnet_features.csv")

if "difficulty_norm" not in df_eff.columns:
    df_eff["difficulty_norm"] = (
        (df_eff["difficulty"] - df_eff["difficulty"].min()) /
        (df_eff["difficulty"].max() - df_eff["difficulty"].min())
    )

if "resolved_img_path" in df_eff.columns:
    df_eff = df_eff[df_eff["resolved_img_path"].notna()].copy()

eff_cols = [c for c in df_eff.columns if c.startswith("effnet_feat_")]
X = df_eff[eff_cols].fillna(0)

threshold = df_eff["difficulty_norm"].median()
y = (df_eff["difficulty_norm"] >= threshold).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline([
    ("pca", PCA(random_state=42)),
    ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))
])

param_grid = {
    "pca__n_components": [3, 5, 10, 20, 30, 50, 100],
    "clf__C": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Лучшие параметры:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

print("\n=== EfficientNet после подбора ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Лучшие параметры: {'clf__C': 0.01, 'pca__n_components': 50}
Best CV F1: 0.6121659030624548

=== EfficientNet после подбора ===
Accuracy: 0.48484848484848486
F1-score: 0.48484848484848486
Balanced accuracy: 0.48484848484848486

Classification report:
              precision    recall  f1-score   support

           0       0.48      0.48      0.48        33
           1       0.48      0.48      0.48        33

    accuracy                           0.48        66
   macro avg       0.48      0.48      0.48        66
weighted avg       0.48      0.48      0.48        66

Confusion matrix:
[[16 17]
 [17 16]]


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

# =========================================================
# 1. Загружаем оба датасета
# =========================================================
df_old = pd.read_csv("data_with_image_features_fixed.csv")
df_eff = pd.read_csv("data_with_efficientnet_features.csv")

# Нормализуем difficulty_norm, если вдруг нет
for d in [df_old, df_eff]:
    if "difficulty_norm" not in d.columns:
        d["difficulty_norm"] = (
            (d["difficulty"] - d["difficulty"].min()) /
            (d["difficulty"].max() - d["difficulty"].min())
        )

# Оставляем только строки с найденной картинкой
if "resolved_img_path" in df_old.columns:
    df_old = df_old[df_old["resolved_img_path"].notna()].copy()
if "resolved_img_path" in df_eff.columns:
    df_eff = df_eff[df_eff["resolved_img_path"].notna()].copy()

# =========================================================
# 2. Берём общую основу и склеиваем признаки
# =========================================================
# Базовые колонки для объединения
base_cols = ["answer", "difficulty", "difficulty_norm", "img_url", "resolved_img_path"]
base_cols = [c for c in base_cols if c in df_old.columns and c in df_eff.columns]

img_cols = [c for c in df_old.columns if c.startswith("img_feat_")]
eff_cols = [c for c in df_eff.columns if c.startswith("effnet_feat_")]

df_combo = pd.concat(
    [
        df_old[base_cols + img_cols].reset_index(drop=True),
        df_eff[eff_cols].reset_index(drop=True)
    ],
    axis=1
)

# =========================================================
# 3. Бинарная цель по медиане
# =========================================================
threshold = df_combo["difficulty_norm"].median()
y = (df_combo["difficulty_norm"] >= threshold).astype(int)

# Один и тот же split для всех сравнений
X_dummy = df_combo[[img_cols[0]]].copy()  # просто для индексов split
X_train_idx, X_test_idx, y_train, y_test = train_test_split(
    X_dummy.index, y, test_size=0.2, random_state=42, stratify=y
)

# =========================================================
# 4. Функция быстрого теста
# =========================================================
def evaluate_feature_set(name, X, n_components):
    X_train = X.loc[X_train_idx].fillna(0)
    X_test = X.loc[X_test_idx].fillna(0)

    model = Pipeline([
        ("pca", PCA(n_components=n_components, random_state=42)),
        ("clf", LogisticRegression(
            C=0.1,
            max_iter=5000,
            class_weight="balanced"
        ))
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    return {
        "features": name,
        "pca": n_components,
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred)
    }

# =========================================================
# 5. Быстрые эксперименты
# =========================================================
results = []

# только старые признаки
results.append(evaluate_feature_set(
    "img_feat only",
    df_combo[img_cols],
    n_components=5
))

# только EfficientNet
results.append(evaluate_feature_set(
    "effnet_feat only",
    df_combo[eff_cols],
    n_components=5
))

# комбинация обоих наборов — сильное сжатие
combo_cols = img_cols + eff_cols
results.append(evaluate_feature_set(
    "img_feat + effnet_feat",
    df_combo[combo_cols],
    n_components=10
))

# комбинация обоих наборов — чуть больше компонент
results.append(evaluate_feature_set(
    "img_feat + effnet_feat",
    df_combo[combo_cols],
    n_components=20
))

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
print(results_df)


                 features  pca  accuracy        f1  balanced_accuracy
0           img_feat only    5  0.636364  0.666667           0.636364
2  img_feat + effnet_feat   10  0.590909  0.608696           0.590909
1        effnet_feat only    5  0.560606  0.579710           0.560606
3  img_feat + effnet_feat   20  0.545455  0.571429           0.545455


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

proba = best_model.predict_proba(X_test)[:, 1]

best_thr = 0
best_f1 = 0

for t in np.linspace(0.3, 0.7, 100):
    y_pred = (proba >= t).astype(int)
    f1 = f1_score(y_test, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_thr = t

print("Лучший threshold:", best_thr)
print("Лучший F1:", best_f1)

Лучший threshold: 0.3
Лучший F1: 0.6595744680851063


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# =========================================
# 1. Подготовка данных
# =========================================
df_ens = df.copy()

if "difficulty_norm" not in df_ens.columns:
    df_ens["difficulty_norm"] = (
        (df_ens["difficulty"] - df_ens["difficulty"].min()) /
        (df_ens["difficulty"].max() - df_ens["difficulty"].min())
    )

if "resolved_img_path" in df_ens.columns:
    df_ens = df_ens[df_ens["resolved_img_path"].notna()].copy()

img_cols = [c for c in df_ens.columns if c.startswith("img_feat_")]
X = df_ens[img_cols].fillna(0)

threshold = df_ens["difficulty_norm"].median()
y = (df_ens["difficulty_norm"] >= threshold).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================================
# 2. Модели
# =========================================
logreg_model = Pipeline([
    ("pca", PCA(n_components=5, random_state=42)),
    ("clf", LogisticRegression(
        C=0.1,
        max_iter=5000,
        class_weight="balanced"
    ))
])

svc_model = Pipeline([
    ("pca", PCA(n_components=5, random_state=42)),
    ("clf", SVC(
        C=0.1,
        kernel="rbf",
        probability=True,
        class_weight="balanced"
    ))
])

# =========================================
# 3. Обучение
# =========================================
logreg_model.fit(X_train, y_train)
svc_model.fit(X_train, y_train)

# =========================================
# 4. Вероятности и ансамбль
# =========================================
proba_logreg = logreg_model.predict_proba(X_test)[:, 1]
proba_svc = svc_model.predict_proba(X_test)[:, 1]

# простое среднее
proba_ensemble = (proba_logreg + proba_svc) / 2

# порог 0.5
y_pred_ensemble = (proba_ensemble >= 0.5).astype(int)

# =========================================
# 5. Метрики ансамбля
# =========================================
print("=== Ансамбль: LogisticRegression + SVC ===")
print("Accuracy:", accuracy_score(y_test, y_pred_ensemble))
print("F1-score:", f1_score(y_test, y_pred_ensemble))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_pred_ensemble))

print("\nClassification report:")
print(classification_report(y_test, y_pred_ensemble, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_ensemble))

=== Ансамбль: LogisticRegression + SVC ===
Accuracy: 0.6363636363636364
F1-score: 0.6666666666666666
Balanced accuracy: 0.6363636363636364

Classification report:
              precision    recall  f1-score   support

           0       0.67      0.55      0.60        33
           1       0.62      0.73      0.67        33

    accuracy                           0.64        66
   macro avg       0.64      0.64      0.63        66
weighted avg       0.64      0.64      0.63        66

Confusion matrix:
[[18 15]
 [ 9 24]]


# Предсказать

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# =========================================================
# 1. Загружаем оба датасета
# =========================================================
df_old = pd.read_csv("data_with_image_features_fixed.csv")
df_eff = pd.read_csv("data_with_efficientnet_features.csv")

# =========================================================
# 2. Приводим к единому виду
# =========================================================
common_cols = ["answer", "difficulty", "img_url", "resolved_img_path", "solved", "started"]
common_cols = [c for c in common_cols if c in df_old.columns and c in df_eff.columns]

img_cols = [c for c in df_old.columns if c.startswith("img_feat_")]
eff_cols = [c for c in df_eff.columns if c.startswith("effnet_feat_")]

# склеиваем
df = pd.concat(
    [
        df_old[common_cols + img_cols].reset_index(drop=True),
        df_eff[eff_cols].reset_index(drop=True)
    ],
    axis=1
)

print("Размер после объединения:", df.shape)

# =========================================================
# 3. Чистка данных
# =========================================================
df = df[df["started"] > 0].copy()

# =========================================================
# 4. Считаем solve_rate
# =========================================================
df["solve_rate"] = df["solved"] / df["started"]

# =========================================================
# 5. Сглаживание (КЛЮЧЕВОЙ ШАГ)
# =========================================================
global_mean = df["solve_rate"].mean()

df["solve_rate_smooth"] = (
    df["solved"] + global_mean * 50
) / (df["started"] + 50)

print("\nСтатистика solve_rate:")
print(df["solve_rate"].describe())

print("\nСтатистика solve_rate_smooth:")
print(df["solve_rate_smooth"].describe())

# =========================================================
# 6. Признаки и таргет
# =========================================================
# ВАЖНО: используем только лучшие признаки
X = df[img_cols].fillna(0)

# сравним два таргета
y_raw = df["solve_rate"]
y_smooth = df["solve_rate_smooth"]

# =========================================================
# 7. Train/test split
# =========================================================
X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X, y_raw, test_size=0.2, random_state=42
)

_, _, y_train_smooth, y_test_smooth = train_test_split(
    X, y_smooth, test_size=0.2, random_state=42
)

# =========================================================
# 8. Модель
# =========================================================
pipe = Pipeline([
    ("pca", PCA(random_state=42)),
    ("ridge", Ridge())
])

param_grid = {
    "pca__n_components": [3, 5, 10, 20],
    "ridge__alpha": [0.01, 0.1, 1, 10, 100]
}

# =========================================================
# 9. Обучение на RAW solve_rate
# =========================================================
grid_raw = GridSearchCV(
    pipe, param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1
)

grid_raw.fit(X_train, y_train_raw)

y_pred_raw = grid_raw.best_estimator_.predict(X_test)

print("\n=== RAW solve_rate ===")
print("Лучшие параметры:", grid_raw.best_params_)
print("R2:", r2_score(y_test_raw, y_pred_raw))
print("MAE:", mean_absolute_error(y_test_raw, y_pred_raw))
print("RMSE:", np.sqrt(mean_squared_error(y_test_raw, y_pred_raw)))

# =========================================================
# 10. Обучение на SMOOTH solve_rate
# =========================================================
grid_smooth = GridSearchCV(
    pipe, param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1
)

grid_smooth.fit(X_train, y_train_smooth)

y_pred_smooth = grid_smooth.best_estimator_.predict(X_test)

print("\n=== SMOOTH solve_rate ===")
print("Лучшие параметры:", grid_smooth.best_params_)
print("R2:", r2_score(y_test_smooth, y_pred_smooth))
print("MAE:", mean_absolute_error(y_test_smooth, y_pred_smooth))
print("RMSE:", np.sqrt(mean_squared_error(y_test_smooth, y_pred_smooth)))

Размер после объединения: (327, 1798)

Статистика solve_rate:
count    326.000000
mean       0.954553
std        0.084104
min        0.477876
25%        0.969189
50%        0.987135
75%        0.992966
max        1.000000
Name: solve_rate, dtype: float64

Статистика solve_rate_smooth:
count    326.000000
mean       0.956256
std        0.075337
min        0.529222
25%        0.968456
50%        0.985585
75%        0.991062
max        0.998411
Name: solve_rate_smooth, dtype: float64

=== RAW solve_rate ===
Лучшие параметры: {'pca__n_components': 3, 'ridge__alpha': 100}
R2: 0.03179323604052897
MAE: 0.049226638131983504
RMSE: 0.07364813679984575

=== SMOOTH solve_rate ===
Лучшие параметры: {'pca__n_components': 3, 'ridge__alpha': 100}
R2: 0.03491594266559672
MAE: 0.04430022763091357
RMSE: 0.06575873616222302


In [ ]:
for col in df_old.columns:
    print(col)

id
answer
description
date_posted
img_url
started
solved
users_with_hints
total_hints
solve_rate
hint_usage
avg_hints
difficulty
resolved_img_path
img_feat_0
img_feat_1
img_feat_2
img_feat_3
img_feat_4
img_feat_5
img_feat_6
img_feat_7
img_feat_8
img_feat_9
img_feat_10
img_feat_11
img_feat_12
img_feat_13
img_feat_14
img_feat_15
img_feat_16
img_feat_17
img_feat_18
img_feat_19
img_feat_20
img_feat_21
img_feat_22
img_feat_23
img_feat_24
img_feat_25
img_feat_26
img_feat_27
img_feat_28
img_feat_29
img_feat_30
img_feat_31
img_feat_32
img_feat_33
img_feat_34
img_feat_35
img_feat_36
img_feat_37
img_feat_38
img_feat_39
img_feat_40
img_feat_41
img_feat_42
img_feat_43
img_feat_44
img_feat_45
img_feat_46
img_feat_47
img_feat_48
img_feat_49
img_feat_50
img_feat_51
img_feat_52
img_feat_53
img_feat_54
img_feat_55
img_feat_56
img_feat_57
img_feat_58
img_feat_59
img_feat_60
img_feat_61
img_feat_62
img_feat_63
img_feat_64
img_feat_65
img_feat_66
img_feat_67
img_feat_68
img_feat_69
img_feat_70
img_feat_71

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df_h = df_old.copy()

# защита
df_h = df_h[df_h["solved"] > 0].copy()

# таргет
df_h["target"] = np.log1p(df_h["avg_hints"])

# признаки
img_cols = [c for c in df_h.columns if c.startswith("img_feat_")]
X = df_h[img_cols].fillna(0)
y = df_h["target"]

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# модель
pipe = Pipeline([
    ("pca", PCA(random_state=42)),
    ("ridge", Ridge())
])

param_grid = {
    "pca__n_components": [3, 5, 10, 20],
    "ridge__alpha": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

y_pred = grid.best_estimator_.predict(X_test)

print("Лучшие параметры:", grid.best_params_)
print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

Лучшие параметры: {'pca__n_components': 3, 'ridge__alpha': 100}
R2: 0.023375474974156152
MAE: 0.1882479305077248
RMSE: 0.2261139829886517


# Чужой датасет

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("TrishanuDas/rebus-dataset")
print(ds)

df = ds["train"].to_pandas()
print(df.shape)
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

RebusPuzzlesFullDataset.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1354 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['img_url', 'Difficulty', 'answer', 'hint', 'source', 'is the hint correct?', 'number of units of reasoning need to solve the rebus puzzle', 'text present or not in image', 'language of text in image', 'spelling of individual objects/components DIFFERENT from that in the final caption?', 'COLOR of objects/text in image needed for answering?', 'POSITION of objects/text in image needed for answering?', 'ORIENTATION of objects/text in image needed for answering?', 'ASPECT RATIO of objects/text in image needed for answering?', 'SIZE of objects/text in image needed for answering?', 'STYLE/TEXTURE of objects/text in image needed for answering?', 'NUMBER of objects/text in image needed for answering?', 'is_augmented'],
        num_rows: 1354
    })
})
(1354, 18)


,img_url,Difficulty,answer,hint,source,is the hint correct?,number of units of reasoning need to solve the rebus puzzle,text present or not in image,language of text in image,spelling of individual objects/components DIFFERENT from that in the final caption?,COLOR of objects/text in image needed for answering?,POSITION of objects/text in image needed for answering?,ORIENTATION of objects/text in image needed for answering?,ASPECT RATIO of objects/text in image needed for answering?,SIZE of objects/text in image needed for answering?,STYLE/TEXTURE of objects/text in image needed for answering?,NUMBER of objects/text in image needed for answering?,is_augmented
0,https://raw.githubusercontent.com/abhi1nandy2/...,HARD,For once in my life,four ones in my life,https://eslvault.com/free-printable-rebus-puzz...,1.0,3.0,1.0,ONLY ENGLISH,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,False
1,https://raw.githubusercontent.com/abhi1nandy2/...,EASY,Forget it,None,https://eslvault.com/free-printable-rebus-puzz...,NaN,2.0,1.0,ONLY ENGLISH,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,False
2,https://raw.githubusercontent.com/abhi1nandy2/...,EASY,Try to understand,None,https://eslvault.com/free-printable-rebus-puzz...,NaN,3.0,1.0,ONLY ENGLISH,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,False
3,https://raw.githubusercontent.com/abhi1nandy2/...,EASY,Travel overseas|overseas travel,None,https://eslvault.com/free-printable-rebus-puzz...,NaN,3.0,1.0,ONLY ENGLISH,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,False
4,https://raw.githubusercontent.com/abhi1nandy2/...,EASY,Breakfast,None,https://eslvault.com/free-printable-rebus-puzz...,NaN,2.0,1.0,ONLY ENGLISH,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,False


In [ ]:
print(df.columns.tolist())
print()
print(df.info())

['img_url', 'Difficulty', 'answer', 'hint', 'source', 'is the hint correct?', 'number of units of reasoning need to solve the rebus puzzle', 'text present or not in image', 'language of text in image', 'spelling of individual objects/components DIFFERENT from that in the final caption?', 'COLOR of objects/text in image needed for answering?', 'POSITION of objects/text in image needed for answering?', 'ORIENTATION of objects/text in image needed for answering?', 'ASPECT RATIO of objects/text in image needed for answering?', 'SIZE of objects/text in image needed for answering?', 'STYLE/TEXTURE of objects/text in image needed for answering?', 'NUMBER of objects/text in image needed for answering?', 'is_augmented']

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1354 entries, 0 to 1353
Data columns (total 18 columns):
 #   Column                                                                               Non-Null Count  Dtype  
---  ------                                              

In [ ]:
# Посмотреть распределение целевой переменной
print(df["Difficulty"].value_counts(dropna=False))
print()
print(df["Difficulty"].value_counts(normalize=True, dropna=False))

Difficulty
EASY    713
HARD    628
None     13
Name: count, dtype: int64

Difficulty
EASY    0.526588
HARD    0.463811
None    0.009601
Name: proportion, dtype: float64


In [ ]:
# Быстрые описательные статистики для числовых признаков
num_cols = df.select_dtypes(include=["number", "bool"]).columns.tolist()
print(num_cols)
df[num_cols].describe().T

['is the hint correct?', 'number of units of reasoning need to solve the rebus puzzle', 'text present or not in image', 'spelling of individual objects/components DIFFERENT from that in the final caption?', 'COLOR of objects/text in image needed for answering?', 'POSITION of objects/text in image needed for answering?', 'ORIENTATION of objects/text in image needed for answering?', 'ASPECT RATIO of objects/text in image needed for answering?', 'SIZE of objects/text in image needed for answering?', 'STYLE/TEXTURE of objects/text in image needed for answering?', 'NUMBER of objects/text in image needed for answering?', 'is_augmented']


,count,mean,std,min,25%,50%,75%,max
is the hint correct?,622.0,0.480707,0.500030,0.0,0.0,0.0,1.0,1.0
number of units of reasoning need to solve the rebus puzzle,1325.0,2.248302,0.699217,1.0,2.0,2.0,3.0,9.0
text present or not in image,1328.0,0.910392,0.285728,0.0,1.0,1.0,1.0,1.0
spelling of individual objects/components DIFFERENT from that in the final caption?,1326.0,0.334842,0.472113,0.0,0.0,0.0,1.0,1.0
COLOR of objects/text in image needed for answering?,1327.0,0.010550,0.102209,0.0,0.0,0.0,0.0,1.0
POSITION of objects/text in image needed for answering?,1327.0,0.504898,0.500164,0.0,0.0,1.0,1.0,1.0
ORIENTATION of objects/text in image needed for answering?,1327.0,0.081387,0.273531,0.0,0.0,0.0,0.0,1.0
ASPECT RATIO of objects/text in image needed for answering?,1327.0,0.009043,0.094699,0.0,0.0,0.0,0.0,1.0
SIZE of objects/text in image needed for answering?,1327.0,0.029390,0.168960,0.0,0.0,0.0,0.0,1.0
STYLE/TEXTURE of objects/text in image needed for answering?,1327.0,0.126601,0.332651,0.0,0.0,0.0,0.0,1.0


In [ ]:
df = df.copy()

df["target"] = (df["Difficulty"].str.upper() == "HARD").astype(int)

print("EASY =", (df["target"] == 0).sum())
print("HARD =", (df["target"] == 1).sum())

EASY = 726
HARD = 628


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Числовые/булевы признаки
feature_cols = [
    c for c in df.columns
    if c not in ["Difficulty", "target", "img_url", "answer", "hint", "source", "language of text in image"]
]

# Оставим только числовые и булевы
X = df[feature_cols].copy()
X = X.select_dtypes(include=["number", "bool"])
y = df["target"]

print("Используемые табличные признаки:")
print(X.columns.tolist())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model_tab = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))
])

model_tab.fit(X_train, y_train)
y_pred = model_tab.predict(X_test)

print("=== Табличные признаки ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Используемые табличные признаки:
['is the hint correct?', 'number of units of reasoning need to solve the rebus puzzle', 'text present or not in image', 'spelling of individual objects/components DIFFERENT from that in the final caption?', 'COLOR of objects/text in image needed for answering?', 'POSITION of objects/text in image needed for answering?', 'ORIENTATION of objects/text in image needed for answering?', 'ASPECT RATIO of objects/text in image needed for answering?', 'SIZE of objects/text in image needed for answering?', 'STYLE/TEXTURE of objects/text in image needed for answering?', 'NUMBER of objects/text in image needed for answering?', 'is_augmented']
=== Табличные признаки ===
Accuracy: 0.6715867158671587
F1: 0.6763636363636364

Classification report:
              precision    recall  f1-score   support

           0       0.73      0.61      0.67       145
           1       0.62      0.74      0.68       126

    accuracy                           0.67       271
   macr

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

X_text = df["answer"].fillna("")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

model_text = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=300, ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))
])

model_text.fit(X_train, y_train)
y_pred = model_text.predict(X_test)

print("=== Только answer ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

=== Только answer ===
Accuracy: 0.6383763837638377
F1: 0.5625

Classification report:
              precision    recall  f1-score   support

           0       0.64      0.76      0.69       145
           1       0.64      0.50      0.56       126

    accuracy                           0.64       271
   macro avg       0.64      0.63      0.63       271
weighted avg       0.64      0.64      0.63       271

Confusion matrix:
[[110  35]
 [ 63  63]]


In [ ]:
import os
import requests
import numpy as np
from PIL import Image
from io import BytesIO
from tqdm import tqdm

import torch
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

# Папка для локального кэша
os.makedirs("rebus_images", exist_ok=True)

def download_image(url, out_dir="rebus_images"):
    filename = os.path.join(out_dir, url.split("/")[-1])
    if os.path.exists(filename):
        return filename

    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        with open(filename, "wb") as f:
            f.write(r.content)
        return filename
    except Exception:
        return None

# Скачиваем
local_paths = []
for url in tqdm(df["img_url"], desc="Downloading images"):
    local_paths.append(download_image(url))

df["local_img_path"] = local_paths
print("Не скачалось:", df["local_img_path"].isna().sum())

Не скачалось: 0


In [ ]:
weights = ResNet18_Weights.DEFAULT
base_model = resnet18(weights=weights)
model_img = torch.nn.Sequential(*list(base_model.children())[:-1])
model_img.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=weights.transforms().mean, std=weights.transforms().std)
])

def extract_features(path):
    if pd.isna(path):
        return np.zeros(512, dtype=np.float32)
    try:
        img = Image.open(path).convert("RGB")
        x = transform(img).unsqueeze(0)
        with torch.no_grad():
            feats = model_img(x).squeeze().numpy()
        return feats.astype(np.float32)
    except Exception:
        return np.zeros(512, dtype=np.float32)

img_features = []
for path in tqdm(df["local_img_path"], desc="Extracting image features"):
    img_features.append(extract_features(path))

img_features = np.vstack(img_features)
print(img_features.shape)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 127MB/s]
Extracting image features: 100%|██████████| 1354/1354 [02:19<00:00,  9.72it/s]

(1354, 512)


In [ ]:
from sklearn.decomposition import PCA

X_img = pd.DataFrame(img_features, columns=[f"img_feat_{i}" for i in range(img_features.shape[1])])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X_img, y, test_size=0.2, random_state=42, stratify=y
)

model_img_clf = Pipeline([
    ("pca", PCA(n_components=10, random_state=42)),
    ("clf", LogisticRegression(max_iter=5000, class_weight="balanced"))
])

model_img_clf.fit(X_train, y_train)
y_pred = model_img_clf.predict(X_test)

print("=== Только изображение ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

=== Только изображение ===
Accuracy: 0.4944649446494465
F1: 0.43621399176954734

Classification report:
              precision    recall  f1-score   support

           0       0.53      0.56      0.54       145
           1       0.45      0.42      0.44       126

    accuracy                           0.49       271
   macro avg       0.49      0.49      0.49       271
weighted avg       0.49      0.49      0.49       271

Confusion matrix:
[[81 64]
 [73 53]]


In [ ]:
results = []

def add_result(name, y_true, y_pred):
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred)
    })

# tabular
X = df[feature_cols].copy().select_dtypes(include=["number", "bool"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model_tab.fit(X_train, y_train)
add_result("tabular only", y_test, model_tab.predict(X_test))

# answer
X_text = df["answer"].fillna("")
X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)
model_text.fit(X_train, y_train)
add_result("answer only", y_test, model_text.predict(X_test))

# image
X_img = pd.DataFrame(img_features, columns=[f"img_feat_{i}" for i in range(img_features.shape[1])])
X_train, X_test, y_train, y_test = train_test_split(
    X_img, y, test_size=0.2, random_state=42, stratify=y
)
model_img_clf.fit(X_train, y_train)
add_result("image only", y_test, model_img_clf.predict(X_test))

pd.DataFrame(results).sort_values("f1", ascending=False)

,model,accuracy,f1
0,tabular only,0.671587,0.676364
1,answer only,0.638376,0.562500
2,image only,0.494465,0.436214
